In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../../ecommerce_dataset_updated.csv")

print(df["Category"].unique())                      # check exact spelling
df = df[df["Category"] != "Home and Kitchen"].copy()   # removed from everything below

top_cat = df["Category"].value_counts().idxmax()
top_pay = df["Payment_Method"].value_counts().idxmax()

flows = df.groupby(["Category", "Payment_Method"]).size().reset_index(name="orders")

<StringArray>
[        'Sports',       'Clothing',           'Toys',         'Beauty',
          'Books', 'Home & Kitchen',    'Electronics']
Length: 7, dtype: str


## Sankey diagram

In [3]:
import plotly.graph_objects as go

cats = df["Category"].value_counts().index.tolist()          # biggest first
pays = df["Payment_Method"].value_counts().index.tolist()
labels = [f"{c} ({(df['Category']==c).sum()})" for c in cats] + \
         [f"{p} ({(df['Payment_Method']==p).sum()})" for p in pays]

cat_idx = {c: i for i, c in enumerate(cats)}
pay_idx = {p: i + len(cats) for i, p in enumerate(pays)}

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=20, thickness=20,
              color=["crimson" if c == top_cat else "steelblue" for c in cats] +
                    ["darkorange" if p == top_pay else "seagreen" for p in pays]),
    link=dict(source=flows["Category"].map(cat_idx),
              target=flows["Payment_Method"].map(pay_idx),
              value=flows["orders"],
              color=["rgba(220,20,60,0.35)" if c == top_cat else "rgba(150,150,150,0.3)"
                     for c in flows["Category"]])
))
fig.update_layout(title="Orders: Category → Payment Method (red = top category)", height=600)
fig.show()